# 04 — Debt open items and reconciliation

**Pregunta humana:** ¿Quién le debe a quién, cuánto, por qué, y qué tan reconciliado está?

Este reporte separa stock y flow de deuda interna.  
No mezcla deuda con OPEX ni con resultado operativo.

Foco especial de QA extendido:

- engine closed but ledger open;
- `closed_at < opened_at`;
- repagos aplicados;
- ajustes residuales;
- action list para corregir ledger source;
- debtor/creditor visibles como columnas cuando sea posible.


In [1]:

from pathlib import Path
import sys
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 220)
pd.set_option("display.width", 260)

cwd = Path.cwd().resolve()
repo_root = None
for p in [cwd, *cwd.parents]:
    if (p / "Makefile").exists() and (p / "accounting").is_dir():
        repo_root = p
        break
if repo_root is None:
    raise FileNotFoundError("Could not find repo root. Run from inside accounting-backend.")

reports_dir = repo_root / "accounting" / "notebooks" / "accounting_reports"
if str(reports_dir) not in sys.path:
    sys.path.insert(0, str(reports_dir))

from _shared import *

repo_root = find_repo_root(repo_root)
pack_dir = professional_pack_dir(repo_root)

artifact_inventory = inspect_artifacts(repo_root)
metrics = load_annual_dashboard_metrics(repo_root)
annual_qa = load_annual_dashboard_qa(repo_root)
debt_status = load_debt_status_reconciliation(repo_root)

years = available_years(metrics)
currencies = available_currencies(metrics)
extended_qa = extended_qa_findings(metrics, artifact_inventory, debt_status)

print("repo_root:", repo_root)
print("pack_dir:", pack_dir)
print("years:", years)
print("currencies:", currencies)
print("metric rows:", len(metrics))


repo_root: /home/matias/repos/accounting-backend
pack_dir: /home/matias/repos/accounting-backend/out/professional_pack/latest
years: ['2022', '2023', '2024', '2025', '2026']
currencies: ['ARS', 'N/A', 'USD']
metric rows: 287


## 1. Debt QA extendido

Este bloque reconcilia la lectura de la notebook con los artifacts del engine de deuda.


In [2]:
debt_qa = extended_qa[extended_qa["area"].isin(["debt", "metrics", "currency", "display"])].copy()
display(short_status_table(debt_qa))
export_table(debt_qa, repo_root, "debt_extended_qa.csv")

if debt_status.empty:
    print("No debt_status_reconciliation artifact found.")
else:
    display(debt_status.head(40))


,severity,area,check,n,detail
2,ok,currency,no_cross_currency_totals,0,No suspicious cross-currency Currency labels f...
1,ok,metrics,available_metric_has_value,0,No available metrics with NaN values
6,warning,debt,engine_closed_but_ledger_open,15,15 rows engine closed but ledger still open
4,warning,display,hidden_metric_id_collisions,50,50 display key groups map to multiple metric_i...
5,warning,metrics,unavailable_visible,3,3 unavailable/blocked rows should be surfaced ...


,debt_id,source_tx_id,debtor,creditor,currency,item_type,ledger_status,engine_status,original_amount,open_amount,closed_at,reconciliation_note
0,interes::901ea166e20f85a1,901ea166e20f85a1,Alejandro,MI,USD,Interes,abierto,open,73.0,73.0,NaN,aligned
1,interes::220a4042132a9ddd,220a4042132a9ddd,Alejandro,MI,USD,Interes,abierto,open,5.0,5.0,NaN,aligned
2,interes::cb96be0120cd28d2,cb96be0120cd28d2,Alejandro,MI,USD,Interes,abierto,open,76.0,76.0,NaN,aligned
3,interes::6cd365921e99617d,6cd365921e99617d,Alejandro,MI,USD,Interes,abierto,open,78.0,78.0,NaN,aligned
4,interes::6e8bf574db97e862,6e8bf574db97e862,Alejandro,MI,USD,Interes,abierto,open,6.0,6.0,NaN,aligned
5,prestamo::4353a969f6ff298e,4353a969f6ff298e,Alejandro,MI,USD,Prestamo,abierto,open,93.0,93.0,NaN,aligned
6,prestamo::9b3beb559c9654bb,9b3beb559c9654bb,Alejandro,MI,USD,Prestamo,abierto,open,84.0,84.0,NaN,aligned
7,prestamo::f18e8b5b9d90a053,f18e8b5b9d90a053,Alejandro,MI,USD,Prestamo,abierto,open,3.0,3.0,NaN,aligned
8,prestamo::103cde22422a081a,103cde22422a081a,Alejandro,PM,USD,Prestamo,abierto,closed,120.0,0.0,2026-05-11,engine closed via repayments but ledger is not...
9,prestamo::2174bec7a0b7cf2e,2174bec7a0b7cf2e,Alejandro,PM,USD,Prestamo,abierto,open,97.0,97.0,NaN,aligned


## 2. Stock de deuda abierta

Separación stock:

```text
Deuda total abierta
Principal abierto
Interés abierto
Saldos abiertos por contraparte
```

Estos saldos son obligaciones internas. No son OPEX.


In [3]:
debt_stock_specs = [
    spec("Debt", "Stock total", "Deuda total abierta", "ID.DEBT.TOTAL.OPEN", 100,
         professional_comment="Stock total de deuda interna abierta por moneda."),
    spec("Debt", "Stock total", "Principal abierto", "ID.DEBT.PRINCIPAL.OPEN", 110,
         professional_comment="Componente principal de la deuda abierta."),
    spec("Debt", "Stock total", "Interés abierto", "ID.DEBT.INTEREST.OPEN", 120,
         professional_comment="Componente interés/costo de oportunidad si corresponde."),
    spec("Debt", "Por contraparte", "Saldo abierto por contraparte", "ID.DEBT.OPEN.BY_COUNTERPARTY", 200,
         append_dimension=True,
         professional_comment="Quién le debe a quién. Conviene promover debtor/creditor a columnas en iteración posterior."),
    spec("Debt", "Posición", "Posición neta PM", "ID.DEBT.NET_PM_POSITION", 300,
         professional_comment="Debe interpretarse solo si la definición de entidad PM está cerrada."),
]

debt_stock_table = build_statement_table(metrics, debt_stock_specs, years, include_debug_cols=True, drop_all_empty_years=False)
display_statement(debt_stock_table, "Stock anual de deuda interna", "Saldos abiertos por moneda y contraparte. Debug cols visibles para revisar etiquetas.", debug=True)
export_table(debt_stock_table, repo_root, "debt_stock_summary.csv")


## Stock anual de deuda interna

Saldos abiertos por moneda y contraparte. Debug cols visibles para revisar etiquetas.

section,line,metric_id,dimension_name,dimension_value,Currency,2022,2023,2024,2025,2026,value_status,professional_comment,caveat,source_table,format_hint
Por contraparte,Saldo abierto por contraparte — debtor_creditor: Alejandro -> MI,ID.DEBT.OPEN.BY_COUNTERPARTY,debtor_creditor,Alejandro -> MI,USD,s/d,s/d,253,334,418,available,Quién le debe a quién. Conviene promover debtor/creditor a columnas en iteración posterior.,,monthly_debt_position.csv,number
Por contraparte,Saldo abierto por contraparte — debtor_creditor: Alejandro -> PM,ID.DEBT.OPEN.BY_COUNTERPARTY,debtor_creditor,Alejandro -> PM,USD,s/d,6.703,6.703,6.703,6.514,available,Quién le debe a quién. Conviene promover debtor/creditor a columnas en iteración posterior.,,monthly_debt_position.csv,number
Por contraparte,Saldo abierto por contraparte — debtor_creditor: Hector -> MI,ID.DEBT.OPEN.BY_COUNTERPARTY,debtor_creditor,Hector -> MI,USD,s/d,s/d,s/d,490,505,available,Quién le debe a quién. Conviene promover debtor/creditor a columnas en iteración posterior.,,monthly_debt_position.csv,number
Por contraparte,Saldo abierto por contraparte — debtor_creditor: PM -> MI,ID.DEBT.OPEN.BY_COUNTERPARTY,debtor_creditor,PM -> MI,USD,s/d,5.792,8.536,6.803,2.116,available,Quién le debe a quién. Conviene promover debtor/creditor a columnas en iteración posterior.,,monthly_debt_position.csv,number
Por contraparte,Saldo abierto por contraparte — debtor_creditor: PM -> Primos,ID.DEBT.OPEN.BY_COUNTERPARTY,debtor_creditor,PM -> Primos,USD,s/d,897,897,897,897,available,Quién le debe a quién. Conviene promover debtor/creditor a columnas en iteración posterior.,,monthly_debt_position.csv,number
Posición,Posición neta PM,ID.DEBT.NET_PM_POSITION,,,USD,s/d,0,0,0,0,available,Debe interpretarse solo si la definición de entidad PM está cerrada.,,monthly_debt_position.csv,number
Stock total,Deuda total abierta,ID.DEBT.TOTAL.OPEN,,,USD,s/d,13.392,16.389,15.227,10.450,available,Stock total de deuda interna abierta por moneda.,"Debt is stock, not flow; not mixed into operating result. | Debt is stock, not flow; not mixed into operating result. | Debt is stock, not flow; not mixed into operating result.",monthly_debt_position.csv,number
Stock total,Principal abierto,ID.DEBT.PRINCIPAL.OPEN,,,USD,s/d,13.392,16.316,15.073,10.197,available,Componente principal de la deuda abierta.,"Debt is stock, not flow; not mixed into operating result. | Debt is stock, not flow; not mixed into operating result. | Debt is stock, not flow; not mixed into operating result.",monthly_debt_position.csv,number
Stock total,Interés abierto,ID.DEBT.INTEREST.OPEN,,,USD,s/d,0,73,154,253,available,Componente interés/costo de oportunidad si corresponde.,"Debt is stock, not flow; not mixed into operating result. | Debt is stock, not flow; not mixed into operating result. | Debt is stock, not flow; not mixed into operating result.",monthly_debt_position.csv,number


PosixPath('/home/matias/repos/accounting-backend/out/professional_pack/latest/tables/debt_stock_summary.csv')

## 3. Debt roll-forward

Formato contable deseado:

```text
Deuda inicial
+ nuevos claims
+ intereses devengados
- repagos
+/- ajustes residuales
= deuda final
```

Nota: si la métrica de deuda inicial no existe todavía, se derivará más adelante desde cierre del año anterior. Esta notebook muestra los flows disponibles y el cierre.


In [4]:
debt_flow_specs = [
    spec("Debt", "Actividad de deuda", "Nuevos claims / adelantos", "ID.DEBT.ACTIVITY.NEW_CLAIMS", 100,
         append_dimension=True,
         professional_comment="Nuevos préstamos/adelantos reconocidos. No son OPEX."),
    spec("Debt", "Actividad de deuda", "Intereses devengados", "ID.DEBT.ACTIVITY.INTEREST_ACCRUED", 110,
         append_dimension=True,
         professional_comment="Costo de oportunidad/intereses devengados si el engine los produce."),
    spec("Debt", "Actividad de deuda", "Repagos de deuda interna", "ID.DEBT.ACTIVITY.REPAYMENTS", 120,
         append_dimension=True,
         sign=-1,
         professional_comment="Repagos mostrados con signo negativo porque reducen deuda."),
    spec("Debt", "Actividad de deuda", "Ajustes residuales", "ID.DEBT.ACTIVITY.ADJUSTMENTS", 130,
         append_dimension=True,
         professional_comment="Ajustes residuales que requieren explicación cuando son relevantes."),
    spec("Debt", "Actividad de deuda", "Cambio neto de deuda", "ID.DEBT.ACTIVITY.NET_CHANGE", 140,
         append_dimension=True,
         professional_comment="Cambio neto anual de deuda. Debe reconciliar con apertura/cierre."),
    spec("Debt", "Cierre", "Deuda final abierta", "ID.DEBT.TOTAL.OPEN", 200,
         professional_comment="Stock de cierre anual."),
]

debt_rollforward = build_statement_table(metrics, debt_flow_specs, years, include_debug_cols=True, drop_all_empty_years=True)
display_statement(debt_rollforward, "Movimiento anual de deuda interna", "Flows anuales y cierre. Repagos se muestran negativos a nivel presentación.", debug=True)
export_table(debt_rollforward, repo_root, "debt_rollforward.csv")


## Movimiento anual de deuda interna

Flows anuales y cierre. Repagos se muestran negativos a nivel presentación.

section,line,metric_id,dimension_name,dimension_value,Currency,2022,2023,2024,2025,2026,value_status,professional_comment,caveat,source_table,format_hint
Actividad de deuda,Nuevos claims / adelantos — debtor_creditor: Alejandro -> MI,ID.DEBT.ACTIVITY.NEW_CLAIMS,debtor_creditor,Alejandro -> MI,USD,s/d,s/d,180,0,0,available,Nuevos préstamos/adelantos reconocidos. No son OPEX.,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere.,monthly_debt_activity.csv,number
Actividad de deuda,Nuevos claims / adelantos — debtor_creditor: Alejandro -> PM,ID.DEBT.ACTIVITY.NEW_CLAIMS,debtor_creditor,Alejandro -> PM,USD,s/d,6.703,0,0,0,available,Nuevos préstamos/adelantos reconocidos. No son OPEX.,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere.,monthly_debt_activity.csv,number
Actividad de deuda,Nuevos claims / adelantos — debtor_creditor: Hector -> MI,ID.DEBT.ACTIVITY.NEW_CLAIMS,debtor_creditor,Hector -> MI,USD,s/d,s/d,s/d,490,0,available,Nuevos préstamos/adelantos reconocidos. No son OPEX.,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere.,monthly_debt_activity.csv,number
Actividad de deuda,Nuevos claims / adelantos — debtor_creditor: PM -> MI,ID.DEBT.ACTIVITY.NEW_CLAIMS,debtor_creditor,PM -> MI,USD,s/d,5.806,2.750,268,0,available,Nuevos préstamos/adelantos reconocidos. No son OPEX.,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere.,monthly_debt_activity.csv,number
Actividad de deuda,Nuevos claims / adelantos — debtor_creditor: PM -> Primos,ID.DEBT.ACTIVITY.NEW_CLAIMS,debtor_creditor,PM -> Primos,USD,s/d,897,0,0,0,available,Nuevos préstamos/adelantos reconocidos. No son OPEX.,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere.,monthly_debt_activity.csv,number
Actividad de deuda,Intereses devengados — debtor_creditor: Alejandro -> MI,ID.DEBT.ACTIVITY.INTEREST_ACCRUED,debtor_creditor,Alejandro -> MI,USD,s/d,s/d,73,81,84,available,Costo de oportunidad/intereses devengados si el engine los produce.,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere.,monthly_debt_activity.csv,number
Actividad de deuda,Intereses devengados — debtor_creditor: Alejandro -> PM,ID.DEBT.ACTIVITY.INTEREST_ACCRUED,debtor_creditor,Alejandro -> PM,USD,s/d,0,0,0,0,available,Costo de oportunidad/intereses devengados si el engine los produce.,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere.,monthly_debt_activity.csv,number
Actividad de deuda,Intereses devengados — debtor_creditor: Hector -> MI,ID.DEBT.ACTIVITY.INTEREST_ACCRUED,debtor_creditor,Hector -> MI,USD,s/d,s/d,s/d,0,15,available,Costo de oportunidad/intereses devengados si el engine los produce.,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classified elsewhere.,monthly_debt_activity.csv,number
Actividad de deuda,Intereses devengados — debtor_creditor: PM -> MI,ID.DEBT.ACTIVITY.INTEREST_ACCRUED,debtor_creditor,PM -> MI,USD,s/d,0,101,187,200,available,Costo de oportunidad/intereses devengados si el engine los produce.,Debt movement; not OPEX or funding unless classified elsewhere. | Debt movement; not OPEX or funding unless classif

PosixPath('/home/matias/repos/accounting-backend/out/professional_pack/latest/tables/debt_rollforward.csv')

## 4. Action list para ledger source

Este es el cuadro operativo: qué items el engine considera cerrados pero el ledger todavía muestra abiertos.  
La regla que veníamos usando:

- si `opened_at <= closed_at`, `open_amount=0`, `engine_status=closed`, se puede recomendar pasar a `cerrado`;
- si `closed_at < opened_at`, no editar masivamente: mandar a revisión.


In [5]:
actions = debt_action_list(debt_status)
display(actions)
export_table(actions, repo_root, "debt_open_items_action_list.csv")

debt_recon_summary = summarize_debt_reconciliation(debt_status)
display(debt_recon_summary)
export_table(debt_recon_summary, repo_root, "debt_reconciliation_summary.csv")


,debt_id,source_tx_id,closed_at,debtor,creditor,currency,item_type,original_amount,open_amount,ledger_status,engine_status,chronology_status,recommended_status,recommendation_reason
8,prestamo::103cde22422a081a,103cde22422a081a,2026-05-11,Alejandro,PM,USD,Prestamo,120.0,0.0,abierto,closed,unknown,review_required,Engine closed this item but closed_at is befor...
16,prestamo::f2c61eb1dbc0dbff,f2c61eb1dbc0dbff,2026-05-11,Alejandro,PM,USD,Prestamo,10.0,0.0,abierto,closed,unknown,review_required,Engine closed this item but closed_at is befor...
17,prestamo::06c13c472d63d00b,06c13c472d63d00b,2026-05-11,Alejandro,PM,USD,Prestamo,50.0,0.0,abierto,closed,unknown,review_required,Engine closed this item but closed_at is befor...
26,prestamo::97fa30e9b3192493,97fa30e9b3192493,2026-05-11,Alejandro,PM,USD,Prestamo,5.0,0.0,abierto,closed,unknown,review_required,Engine closed this item but closed_at is befor...
27,prestamo::b606434ff2d772ed,b606434ff2d772ed,2026-05-11,Alejandro,PM,USD,Prestamo,4.0,0.0,abierto,closed,unknown,review_required,Engine closed this item but closed_at is befor...
44,interes::dfc3d7c7bbb9201b,dfc3d7c7bbb9201b,2023-11-07,PM,MI,USD,Interes,101.0,0.0,abierto,closed,unknown,review_required,Engine closed this item but closed_at is befor...
45,interes::b011815ff9267dbd,b011815ff9267dbd,2023-11-07,PM,MI,USD,Interes,83.0,0.0,abierto,closed,unknown,review_required,Engine closed this item but closed_at is befor...
46,interes::bf0bd40044e6facd,bf0bd40044e6facd,2025-03-20,PM,MI,USD,Interes,104.0,0.0,abierto,closed,unknown,review_required,Engine closed this item but closed_at is befor...
47,interes::ddfa64d11488e39c,ddfa64d11488e39c,2025-03-20,PM,MI,USD,Interes,107.0,0.0,abierto,closed,unknown,review_required,Engine closed this item but closed_at is befor...
48,interes::e12ae354410759d5,e12ae354410759d5,2023-11-07,PM,MI,USD,Interes,8.0,0.0,abierto,closed,unknown,review_required,Engine closed this item but closed_at is befor...


,issue,n_rows
1,aligned,104
2,engine closed via repayments but ledger is not...,15
0,engine_closed_but_ledger_open,15
3,ledger says cerrado but engine still leaves ba...,6


PosixPath('/home/matias/repos/accounting-backend/out/professional_pack/latest/tables/debt_reconciliation_summary.csv')

## 5. Comentarios profesionales

Este bloque convierte el debt engine en una narrativa útil para rendición de cuentas.


In [6]:
debt_comments = pd.DataFrame([
    {"tema": "Separación stock/flow", "comentario": "Deuda final es stock; nuevos claims, intereses, repagos y ajustes son flows. No deben mezclarse con OPEX."},
    {"tema": "PM -> MI", "comentario": "Si aparece como principal movimiento, documenta financiamiento/adelantos de MI al sistema PM y posterior reducción por repagos."},
    {"tema": "Repagos", "comentario": "En presentación conviene mostrarlos negativos porque reducen deuda, aunque el artifact source los registre como monto positivo."},
    {"tema": "Ajustes", "comentario": "Ajustes residuales no son necesariamente errores, pero deben tener explicación contable o engine note."},
    {"tema": "Ledger status", "comentario": "Items engine-closed pero ledger-open deben corregirse con cuidado; closed_at < opened_at requiere investigación antes de edición masiva."},
    {"tema": "Contraparte", "comentario": "Para uso familiar/legal, debtor y creditor deben ser columnas explícitas, no solo texto dentro de la línea."},
])
display(debt_comments)
export_table(debt_comments, repo_root, "debt_professional_comments.csv")


,tema,comentario
0,Separación stock/flow,"Deuda final es stock; nuevos claims, intereses..."
1,PM -> MI,"Si aparece como principal movimiento, document..."
2,Repagos,En presentación conviene mostrarlos negativos ...
3,Ajustes,Ajustes residuales no son necesariamente error...
4,Ledger status,Items engine-closed pero ledger-open deben cor...
5,Contraparte,"Para uso familiar/legal, debtor y creditor deb..."


PosixPath('/home/matias/repos/accounting-backend/out/professional_pack/latest/tables/debt_professional_comments.csv')

## 6. Export report

In [7]:
md_path = write_markdown_report(
    repo_root,
    "04_debt_open_items_and_reconciliation.md",
    "04 — Debt open items and reconciliation",
    [
        ("Professional scope", "Reporte de deuda interna: stock, flow, open items y reconciliación. No es OPEX ni resultado operativo."),
        ("Debt QA", debt_qa),
        ("Debt stock summary", debt_stock_table),
        ("Debt roll-forward", debt_rollforward),
        ("Debt action list", actions),
        ("Debt reconciliation summary", debt_recon_summary),
        ("Professional comments", debt_comments),
    ],
)
html_path = write_html_report(
    repo_root,
    "04_debt_open_items_and_reconciliation.html",
    "04 — Debt open items and reconciliation",
    [
        ("Professional scope", "Reporte de deuda interna: stock, flow, open items y reconciliación. No es OPEX ni resultado operativo."),
        ("Debt QA", debt_qa),
        ("Debt stock summary", debt_stock_table),
        ("Debt roll-forward", debt_rollforward),
        ("Debt action list", actions),
        ("Debt reconciliation summary", debt_recon_summary),
        ("Professional comments", debt_comments),
    ],
)
print("markdown:", md_path.relative_to(repo_root))
print("html:", html_path.relative_to(repo_root))


markdown: out/professional_pack/latest/markdown/04_debt_open_items_and_reconciliation.md
html: out/professional_pack/latest/html/04_debt_open_items_and_reconciliation.html
